# Prompting Craft

*System prompts, XML, few-shot, and output constraints.*

- A prompt that works once interactively often breaks in production against untested inputs.
- The fix is **not more words** — it's identifying which *structural* piece is missing and adding that one piece.
- Rewording changes how you say something. It doesn't add the missing structure.
- Boundary being crossed? Clearer phrasing won't fix it. Format drifting? "Please format correctly" won't fix it.

## Diagnose, don't reword

**The failure mode tells you which technique is absent.** Diagnose first, then add that one technique.

| What you observed | What's missing | Why it's the fix |
|---|---|---|
| Wrong **shape** — a sentence where you expected a label, prose where you expected JSON | **Output constraint** — form, field names, and stopping point were never specified | Controls the *form* of the response independent of its content. Without one you get plausible text the parser wasn't built to accept |
| Content is **off** — scope drifts, tone shifts, answers a wider question; worse deeper into the conversation | **System prompt**, or a more specific one — the behavioral contract was too vague to hold across turns | Sets rules that apply to every response regardless of the user turn. Underspecified = nothing holding role, scope, and format steady |
| Task right, **structure invented** — understood the job, produced a shape you never asked for | **Few-shot examples** — structure can't be inferred from a description alone | Show the pattern rather than describe it. One correct input→output pair pins down a shape words often can't |
| Clean on tested inputs, **breaks on a variant** — edge case, unusual field, input you didn't anticipate | **A constraint covering the variant** — handles the happy path, no rule for the case that breaks the parser | Validated against a narrow set. Naming the variant, or adding an example covering it, closes the gap |

⚠️ **Prompt getting longer every pass = you're skipping the diagnosis step and just adding words.**

In [1]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()

# NOTE: the rest of this course uses claude-sonnet-5; everything here behaves the same
# on it. Classification is a low-effort task — see the extended thinking module for why
# `effort` is dialled down throughout this notebook.
model = "claude-opus-5"

## Worked example: a classifier that returns the wrong shape

**Task:** classify support tickets into `billing`, `technical`, `escalation`.

**First attempt** — a bare instruction, no constraint on the output:

In [3]:
TICKET = "I was charged twice for the same month."

BARE_SYSTEM = "You are a support classifier. Classify the ticket."


def classify(system, user, runs=3):
    """Run the same prompt N times and return the raw text of each response."""
    out = []
    for _ in range(runs):
        response = client.messages.create(
            model=model,
            max_tokens=200,
            system=system,
            output_config={"effort": "low"},  # classification doesn't need deep reasoning
            messages=[{"role": "user", "content": user}],
        )
        out.append(next(b.text for b in response.content if b.type == "text").strip())
    return out


bare = classify(BARE_SYSTEM, f"<ticket>{TICKET}</ticket>")
for i, r in enumerate(bare, 1):
    print(f"run {i}: {r!r}")
print(f"\ndistinct outputs: {len(set(bare))}")

run 1: '**Classification**\n\n| Field | Value |\n|---|---|\n| **Category** | Billing → Duplicate/Incorrect Charge |\n| **Sub-type** | Double-billing for a single billing period |\n| **Priority** | High (customer funds affected) |\n| **Sentiment** | Negative / frustrated |\n| **Suggested action** | Verify transaction records for the stated period'
run 2: '**Note:** No category list or taxonomy was provided in my instructions, so I\'m classifying against a standard support schema. If you have a specific set of categories, send it over and I\'ll re-classify accordingly.\n\n**Ticket:** "I was charged twice for the same month."\n\n| Field | Value |\n|---|---|\n| **Category** | Billing |\n| **Subcategory** | Duplicate charge / Overbilling |\n| **Intent** | Refund request (impl'
run 3: "**Classification**\n\n| Field | Value |\n|---|---|\n| **Category** | Billing |\n| **Sub-category** | Duplicate / Double charge |\n| **Intent** | Refund request (implied) |\n| **Priority** | High — money has le

The classifier understands the task and returns the correct category. But the *form* varies —
`"Billing"` on some runs, `"billing"` on others, sometimes a full sentence like
*"This looks like a billing issue."* A router expecting a fixed label set breaks on that.

**Read it against the table: row 1.** Right content, wrong shape → the missing piece is an
**output constraint**.

Note what the cell above proves and what it doesn't: if your three runs came back identical,
nothing in that prompt *guaranteed* it. The shape is unconstrained either way — that's the
point. It will drift on a run or an input you didn't test.

**The fix.** Adding the constraint pulls in two more techniques, because locking the label
set and showing the format are jobs those do better than a written instruction:

- **Few-shot** — shows the exact label and casing to return.
- **XML tags** — keep the examples separate from the instruction, so Claude doesn't read them as part of the task.

In [4]:
FIXED_SYSTEM = """You are a support classifier. Classify each ticket into exactly one of: \
BILLING, TECHNICAL, ESCALATION. Return only the label. No other text.

<sample_input>My account shows two charges for April.</sample_input>
<ideal_output>BILLING</ideal_output>

<sample_input>The API keeps returning a 429 error.</sample_input>
<ideal_output>TECHNICAL</ideal_output>"""

fixed = classify(FIXED_SYSTEM, f"<ticket>{TICKET}</ticket>")
for i, r in enumerate(fixed, 1):
    print(f"run {i}: {r!r}")
print(f"\ndistinct outputs: {len(set(fixed))}")

run 1: 'BILLING'
run 2: 'BILLING'
run 3: 'BILLING'

distinct outputs: 1


**Three techniques, three distinct jobs:**

| Technique | Job in this prompt |
|---|---|
| **System prompt** | Sets the output contract — exactly one label from a fixed set, nothing else |
| **XML tags** | Mark where each example ends and the next begins, so they aren't read as instruction |
| **Few-shot pairs** | Show the exact casing and format rather than describing it |

Together: consistent enough to route programmatically.

## How much to stack

| | |
|---|---|
| **Stack all four** | Against a clearly defined output contract — well-specified formats, edge cases coverable by examples |
| **Simplify** | Don't add all four to a simple task that needs one. "Summarize this paragraph" needs no few-shot examples or output schema |
| **Diagnose before adding more** | Prompt growing longer each iteration rather than more precise. Re-prompted five times and still wrong? Diagnose the failure type before adding text |

## The four techniques

**System prompts** — carry the behavioral contract for the whole session. Write them once
and treat them as your persistent instruction layer. They define Claude's role, the output
format, and any rules that must not change between conversations.

**XML tags** — mark where one block ends and the next begins, so Claude doesn't read your
examples or data as part of the instruction.

**Few-shot examples** — show the pattern rather than describe it. Claude cannot infer an
exact structure from a description alone.

**Output constraints** — control the form of the response independent of its content: field
names, label set, stopping point.

## The iteration loop

**Diagnose → add the one missing technique → re-run.** Still failing? Diagnose again.

| Failure | Missing |
|---|---|
| Wrong format | Output constraint — shape was never specified |
| Wrong content / scope drift | Underspecified system prompt — contract too vague to hold across conversations |
| Correct task, hallucinated structure | Few-shot examples — structure can't be inferred from description |
| Fine on simple inputs, breaks on edge cases | A constraint covering the variant the parser breaks on |

**The fix is structural, not phrasing.**

## Structured outputs: move output control into the API

Everything above shapes output by **writing instructions and hoping Claude follows them**.
The prompt is a *request* — a model can still return a stray sentence, a wrong field name,
or malformed JSON.

**Structured outputs** removes that gap: hand the API a JSON schema and the model is
constrained **at generation time** to match it.

> **Constrained decoding** — as Claude generates each token, the API only allows tokens that
> keep the output valid against your schema. A response that violates the schema *cannot be
> produced in the first place.*

Two situations, usable alone or together in one request.

### 1. JSON outputs — constrain the final response

`output_config.format` with `type: "json_schema"`. Claude returns valid JSON matching the
schema every time.

**Reach for it when** the model itself produces the structured payload your code consumes —
extracting fields from a ticket, formatting an API response. Removes the parse-and-retry
code you'd otherwise write around every call.

Same classifier as above, now schema-constrained rather than instruction-constrained:

In [5]:
import json

LABEL_SCHEMA = {
    "type": "object",
    "properties": {
        "label": {"type": "string", "enum": ["BILLING", "TECHNICAL", "ESCALATION"]},
        "confidence": {"type": "number"},
    },
    "required": ["label", "confidence"],
    "additionalProperties": False,
}

response = client.messages.create(
    model=model,
    max_tokens=1000,
    system="You are a support classifier.",
    messages=[{"role": "user", "content": f"<ticket>{TICKET}</ticket>"}],
    output_config={
        "effort": "low",
        "format": {"type": "json_schema", "schema": LABEL_SCHEMA},
    },
)

raw = next(b.text for b in response.content if b.type == "text")
print("raw text:", raw)
print("parsed:  ", json.loads(raw))   # no fences to strip, no retry loop

raw text: {"label":"BILLING","confidence":0.98}
parsed:   {'label': 'BILLING', 'confidence': 0.98}


The label can no longer come back as `"Billing"` or a sentence — `enum` makes those tokens
unreachable. Note `effort` and `format` both live inside `output_config`.

### 2. Strict tool use — constrain the inputs Claude passes to your tools

`strict: True` on a tool definition validates the arguments against the input schema
**before your code runs**.

**Reach for it in** agentic loops where a malformed argument would crash the function or
trigger a wrong action.

In [6]:
ESCALATE_TOOL = {
    "name": "escalate_ticket",
    "description": "Escalate a support ticket to a human agent.",
    "strict": True,                      # <- arguments validated against the schema
    "input_schema": {
        "type": "object",
        "properties": {
            "ticket_id": {"type": "string"},
            "severity": {"type": "string", "enum": ["low", "medium", "high"]},
            "reason": {"type": "string"},
        },
        "required": ["ticket_id", "severity", "reason"],
        "additionalProperties": False,   # required by strict
    },
}

response = client.messages.create(
    model=model,
    max_tokens=1000,
    tools=[ESCALATE_TOOL],
    messages=[{"role": "user", "content":
               "Ticket T-4417: customer has been double-charged three months running "
               "and is threatening to cancel. Escalate it."}],
)

for block in response.content:
    if block.type == "tool_use":
        print(block.name, "->", block.input)   # guaranteed to match the schema

escalate_ticket -> {'ticket_id': 'T-4417', 'severity': 'high', 'reason': 'Customer has been double-charged for three consecutive months (recurring billing defect, not a one-off error) and is now threatening to cancel. Requires urgent human review to verify the duplicate charges, issue refunds for all three months, correct the underlying billing configuration to prevent a fourth occurrence, and perform account retention outreach.'}


### Why this belongs in production code, not just the prompt

**Reliability under inputs you did not test.** A prompt-level "return only JSON" holds on
the cases you tried and slips on one you didn't — exactly the failure the classifier above
walked through. A schema constraint doesn't slip: the API enforces it **on every token**
rather than trusting the model to remember.

→ Moves output correctness from something you verify *after the fact* to something the API
rules out *before it happens*.

## Costs — weigh these, don't enable it everywhere by default

| Cost | Detail |
|---|---|
| **First request on a new schema is slower** | The API compiles the schema into a grammar first. Compiled grammars cached **24h from last use** — steady traffic on a stable schema pays once; constantly changing schemas pay repeatedly |
| **Input token count rises** | The API adds a system prompt describing the expected format, billed like any input token. Small per call, worth knowing at volume |
| **A guaranteed schema ≠ guaranteed success** | Two cases still return non-matching output: a **refusal** (`stop_reason: "refusal"`) and a **truncation** (`stop_reason: "max_tokens"`, stopping mid-structure) |
| **Doesn't combine with prefilling** | JSON outputs and prefilling the assistant message are incompatible. Pick the one that fits the task |

**So your code still checks `stop_reason` rather than assuming every response parses:**

In [7]:
# Deliberately too small a max_tokens — the schema is still enforced, but the response
# stops mid-structure. This is why you check stop_reason before parsing.
truncated = client.messages.create(
    model=model,
    max_tokens=16,
    system="You are a support classifier.",
    messages=[{"role": "user", "content": f"<ticket>{TICKET}</ticket>"}],
    output_config={
        "effort": "low",
        "format": {"type": "json_schema", "schema": LABEL_SCHEMA},
    },
)

print("stop_reason:", truncated.stop_reason)

raw = next((b.text for b in truncated.content if b.type == "text"), "")
print("raw text:  ", repr(raw))

if truncated.stop_reason == "end_turn":
    print("parsed:", json.loads(raw))
else:
    print(f"-> not parsed: stop_reason was {truncated.stop_reason!r}, not 'end_turn'")

stop_reason: max_tokens
raw text:   '{"label":"BILLING","confidence":0.'
-> not parsed: stop_reason was 'max_tokens', not 'end_turn'


## Summary

| | |
|---|---|
| **Diagnose first** | The failure mode names the missing technique. Don't reword, don't stack blindly |
| **Four techniques** | System prompt (contract) · XML (boundaries) · few-shot (shape) · output constraints (form) |
| **Prompt-level control** | A *request*. Holds on tested inputs, slips on the ones you didn't test |
| **Schema-level control** | Enforced per token. Use `output_config.format` for responses, `strict` for tool arguments |
| **Still check** | `stop_reason` — refusal and truncation both defeat a valid schema |